# Directed HeteroGATv2 on the Elliptic++ Dataset

This notebook trains a **directed heterogeneous GATv2 model** on the Elliptic++ dataset in `data/raw/elliptic++dataset`.

It uses the full directed graph structure:
- `txs_edgelist.csv` for `transaction -> transaction`
- `TxAddr_edgelist.csv` for `transaction -> address`
- `AddrTx_edgelist.csv` for `address -> transaction`
- `AddrAddr_edgelist.csv` for `address -> address`

Feature design:
- transaction nodes use the original transaction features plus wallet-derived transaction enrichments
- address nodes use wallet-feature summaries and directed structural address features

This notebook is separate from `GCN_elliptic_plus_plus.ipynb` so you can compare model families cleanly.
The transaction nodes use a chronological `70/15/15` split by time step: train `1-34`, validation `35-39`, test `40-49`.
Address supervision follows the first-seen time step of each address so we keep the chronology consistent across node types.


In [ ]:
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch_geometric.data import HeteroData
from torch_geometric.nn import GATv2Conv, HeteroConv

plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'data' / 'raw' / 'elliptic++dataset').exists():
            return candidate
    raise FileNotFoundError('Could not find project root containing data/raw/elliptic++dataset')


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'elliptic++dataset'
FIG_DIR = PROJECT_ROOT / 'outputs' / 'figures' / 'elliptic_plus_plus'
MODEL_DIR = PROJECT_ROOT / 'outputs' / 'models'
FIG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f'Project root: {PROJECT_ROOT}')
print(f'Data dir:     {DATA_DIR}')
print(f'Device:       {DEVICE}')
print(f'PyTorch:      {torch.__version__}')


Project root: /Users/kuijun/Desktop/spring-2026-spec-proj
Data dir:     /Users/kuijun/Desktop/spring-2026-spec-proj/data/raw/elliptic++dataset
Device:       cpu
PyTorch:      2.4.1


## 1. Load and prepare the Elliptic++ tables


In [ ]:
tx_features_raw = pd.read_csv(DATA_DIR / 'txs_features.csv')
tx_classes_raw = pd.read_csv(DATA_DIR / 'txs_classes.csv')
tx_edges_raw = pd.read_csv(DATA_DIR / 'txs_edgelist.csv')
tx_addr_edges_raw = pd.read_csv(DATA_DIR / 'TxAddr_edgelist.csv')
addr_tx_edges_raw = pd.read_csv(DATA_DIR / 'AddrTx_edgelist.csv')
addr_addr_edges_raw = pd.read_csv(DATA_DIR / 'AddrAddr_edgelist.csv')
wallet_features_raw = pd.read_csv(DATA_DIR / 'wallets_features.csv')

summary = pd.DataFrame(
    [
        ('txs_features.csv', tx_features_raw.shape[0], tx_features_raw.shape[1]),
        ('txs_classes.csv', tx_classes_raw.shape[0], tx_classes_raw.shape[1]),
        ('txs_edgelist.csv', tx_edges_raw.shape[0], tx_edges_raw.shape[1]),
        ('TxAddr_edgelist.csv', tx_addr_edges_raw.shape[0], tx_addr_edges_raw.shape[1]),
        ('AddrTx_edgelist.csv', addr_tx_edges_raw.shape[0], addr_tx_edges_raw.shape[1]),
        ('AddrAddr_edgelist.csv', addr_addr_edges_raw.shape[0], addr_addr_edges_raw.shape[1]),
        ('wallets_features.csv', wallet_features_raw.shape[0], wallet_features_raw.shape[1]),
    ],
    columns=['file', 'rows', 'columns'],
)
summary


,file,rows,columns
0,txs_features.csv,203769,184
1,txs_classes.csv,203769,2
2,txs_edgelist.csv,234355,2
3,TxAddr_edgelist.csv,837124,2
4,AddrTx_edgelist.csv,477117,2
5,AddrAddr_edgelist.csv,2868964,2
6,wallets_features.csv,1268260,57


In [ ]:
def clean_columns(columns):
    cleaned = []
    for col in columns:
        col = str(col).strip().replace(' ', '_').replace('-', '_').replace('/', '_')
        col = col.replace('(', '').replace(')', '').replace('__', '_')
        cleaned.append(col)
    return cleaned


LABEL_MAP = {'1': 1, '2': 0, '3': -1, 'unknown': -1, 'illicit': 1, 'licit': 0}
# Chronological 70/15/15 split for transaction supervision.
TRAIN_STEPS = list(range(1, 35))
VAL_STEPS = list(range(35, 40))
TEST_STEPS = list(range(40, 50))

transactions = tx_features_raw.copy()
transactions.columns = clean_columns(transactions.columns)
transactions = transactions.rename(columns={'Time_step': 'time_step'})
transactions['txId'] = pd.to_numeric(transactions['txId'], errors='raise').astype(np.int64)
transactions['time_step'] = pd.to_numeric(transactions['time_step'], errors='raise').astype(np.int64)
feature_columns = [col for col in transactions.columns if col not in ['txId', 'time_step']]
transactions[feature_columns] = transactions[feature_columns].apply(pd.to_numeric, errors='coerce')
transactions[feature_columns] = transactions[feature_columns].fillna(0.0)

labels = tx_classes_raw.copy()
labels['txId'] = pd.to_numeric(labels['txId'], errors='raise').astype(np.int64)
labels['class'] = labels['class'].astype(str)
labels['label'] = labels['class'].map(LABEL_MAP).fillna(-1).astype(np.int64)
transactions = transactions.merge(labels[['txId', 'class', 'label']], on='txId', how='left')
transactions['class'] = transactions['class'].fillna('unknown')
transactions['label'] = transactions['label'].fillna(-1).astype(np.int64)

transaction_edges = tx_edges_raw.copy()
transaction_edges['txId1'] = pd.to_numeric(transaction_edges['txId1'], errors='raise').astype(np.int64)
transaction_edges['txId2'] = pd.to_numeric(transaction_edges['txId2'], errors='raise').astype(np.int64)

tx_addr_edges = tx_addr_edges_raw.rename(columns={'output_address': 'address'})
addr_tx_edges = addr_tx_edges_raw.rename(columns={'input_address': 'address'})
addr_addr_edges = addr_addr_edges_raw.rename(columns={'input_address': 'src_address', 'output_address': 'dst_address'})

wallet_features = wallet_features_raw.copy()
wallet_features.columns = clean_columns(wallet_features.columns)
wallet_features = wallet_features.rename(columns={'Time_step': 'time_step'})
wallet_features['time_step'] = pd.to_numeric(wallet_features['time_step'], errors='raise').astype(np.int64)
wallet_numeric_columns = [col for col in wallet_features.columns if col not in ['address', 'time_step']]
wallet_features[wallet_numeric_columns] = wallet_features[wallet_numeric_columns].apply(pd.to_numeric, errors='coerce')
wallet_features[wallet_numeric_columns] = wallet_features[wallet_numeric_columns].fillna(0.0)

# Keep only known labels in the supervised masks; unlabeled rows remain available as graph context.
train_period_mask = transactions['time_step'].isin(TRAIN_STEPS)
val_period_mask = transactions['time_step'].isin(VAL_STEPS)
test_period_mask = transactions['time_step'].isin(TEST_STEPS)
known_label_mask = transactions['label'].isin([0, 1])

print(f'Transactions: {len(transactions):,}')
print(f'Base transaction features: {len(feature_columns):,}')
print(f'Tx->Tx edges: {len(transaction_edges):,}')
print(f'Tx->Addr edges: {len(tx_addr_edges):,}')
print(f'Addr->Tx edges: {len(addr_tx_edges):,}')
print(f'Addr->Addr edges: {len(addr_addr_edges):,}')
print(f'Wallet numeric features: {len(wallet_numeric_columns):,}')
print(f'Time steps: {transactions["time_step"].min()} to {transactions["time_step"].max()}')


Transactions: 203,769
Base transaction features: 182
Tx->Tx edges: 234,355
Tx->Addr edges: 837,124
Addr->Tx edges: 477,117
Addr->Addr edges: 2,868,964
Wallet numeric features: 55
Time steps: 1 to 49


## 2. Build upgraded transaction and address features

The train/validation/test split is chronological, and the feature engineering below respects that order.
Transaction features are scaled using the training window only, while wallet summaries for addresses are aggregated from the training-period snapshots.

We reuse the richer feature construction from the directed hetero notebook family:
- transaction features are enriched with wallet and address statistics aggregated over linked output addresses
- address features combine training-period wallet summaries with directed structural graph features


In [ ]:
wallet_snapshot_features = (
    wallet_features
    .groupby(['address', 'time_step'], as_index=False)[wallet_numeric_columns]
    .mean()
)

addr_out_degree = addr_addr_edges.groupby('src_address').size().rename('addr_out_degree')
addr_in_degree = addr_addr_edges.groupby('dst_address').size().rename('addr_in_degree')
addr_self_loops = (
    addr_addr_edges.loc[addr_addr_edges['src_address'] == addr_addr_edges['dst_address']]
    .groupby('src_address')
    .size()
    .rename('addr_self_loop_count')
)
addr_inputs_tx_degree = addr_tx_edges.groupby('address').size().rename('addr_to_tx_degree')
addr_receives_tx_degree = tx_addr_edges.groupby('address').size().rename('tx_to_addr_degree')

address_graph_features = (
    pd.concat([
        addr_out_degree,
        addr_in_degree,
        addr_self_loops,
        addr_inputs_tx_degree,
        addr_receives_tx_degree,
    ], axis=1)
    .fillna(0.0)
    .reset_index()
    .rename(columns={'index': 'address', 'src_address': 'address'})
)
address_graph_features['addr_total_degree'] = (
    address_graph_features['addr_out_degree'] + address_graph_features['addr_in_degree']
)
address_graph_feature_columns = [col for col in address_graph_features.columns if col != 'address']

transaction_address_time = (
    tx_addr_edges
    .merge(transactions[['txId', 'time_step']], on='txId', how='inner')
    .merge(
        wallet_snapshot_features[['address', 'time_step'] + wallet_numeric_columns],
        on=['address', 'time_step'],
        how='left',
    )
    .merge(
        address_graph_features[['address'] + address_graph_feature_columns],
        on='address',
        how='left',
    )
)
transaction_address_time['wallet_feature_match'] = (
    transaction_address_time[wallet_numeric_columns].notna().any(axis=1).astype(int)
)
transaction_address_time[address_graph_feature_columns] = transaction_address_time[address_graph_feature_columns].fillna(0.0)

all_address_enrichment_columns = wallet_numeric_columns + address_graph_feature_columns
wallet_feature_aggregates = transaction_address_time.groupby('txId')[all_address_enrichment_columns].agg(['mean', 'max'])
wallet_feature_aggregates.columns = [f'wallet_{name}_{stat}' for name, stat in wallet_feature_aggregates.columns]

wallet_feature_meta = transaction_address_time.groupby('txId').agg(
    linked_output_addresses=('address', 'nunique'),
    matched_wallet_rows=('wallet_feature_match', 'sum'),
)
wallet_feature_meta['wallet_match_ratio'] = (
    wallet_feature_meta['matched_wallet_rows']
    / wallet_feature_meta['linked_output_addresses'].replace(0, np.nan)
)

wallet_transaction_features = (
    wallet_feature_meta
    .join(wallet_feature_aggregates, how='left')
    .fillna(0.0)
    .reset_index()
)
transactions_upgraded = transactions.merge(wallet_transaction_features, on='txId', how='left')
wallet_derived_columns = [
    col for col in transactions_upgraded.columns
    if col.startswith('wallet_') or col in ['linked_output_addresses', 'matched_wallet_rows']
]
transactions_upgraded[wallet_derived_columns] = transactions_upgraded[wallet_derived_columns].fillna(0.0)
upgraded_feature_columns = feature_columns + wallet_derived_columns

wallet_train_address_features = (
    wallet_features.loc[wallet_features['time_step'].isin(TRAIN_STEPS)]
    .groupby('address', as_index=False)[wallet_numeric_columns]
    .mean()
)
wallet_train_address_features = wallet_train_address_features.rename(
    columns={col: f'addr_wallet_{col}' for col in wallet_numeric_columns}
)

all_addresses = pd.Index(pd.concat([
    tx_addr_edges['address'],
    addr_tx_edges['address'],
    addr_addr_edges['src_address'],
    addr_addr_edges['dst_address'],
    wallet_train_address_features['address'],
], ignore_index=True).dropna().unique())

address_frame = pd.DataFrame({'address': all_addresses})
address_frame = address_frame.merge(wallet_train_address_features, on='address', how='left')
address_frame = address_frame.merge(address_graph_features, on='address', how='left')
address_feature_columns = [col for col in address_frame.columns if col != 'address']
address_frame[address_feature_columns] = address_frame[address_feature_columns].fillna(0.0)

print(f'Upgraded transaction features: {len(upgraded_feature_columns):,}')
print(f'Address features: {len(address_feature_columns):,}')
print(f'Address nodes: {len(address_frame):,}')
print(f'Transactions with wallet feature matches: {(transactions_upgraded["matched_wallet_rows"] > 0).sum():,}')


Upgraded transaction features: 307
Address features: 61
Address nodes: 822,942
Transactions with wallet feature matches: 202,804


In [ ]:
tx_feature_frame = (
    transactions_upgraded[upgraded_feature_columns]
    .apply(pd.to_numeric, errors='coerce')
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0.0)
)
address_feature_frame = (
    address_frame[address_feature_columns]
    .apply(pd.to_numeric, errors='coerce')
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0.0)
)

tx_train_mask = transactions_upgraded['time_step'].isin(TRAIN_STEPS).to_numpy()
tx_val_mask = transactions_upgraded['time_step'].isin(VAL_STEPS).to_numpy()
tx_test_mask = transactions_upgraded['time_step'].isin(TEST_STEPS).to_numpy()
tx_known_label_mask = transactions_upgraded['label'].isin([0, 1]).to_numpy()

if tx_feature_frame.shape[1] == 0:
    raise ValueError('No transaction feature columns were found for the heterogeneous graph.')
if address_feature_frame.shape[1] == 0:
    raise ValueError('No address feature columns were found for the heterogeneous graph.')
if not tx_train_mask.any():
    raise ValueError('No transaction rows fall inside the configured training time steps.')

hetero_tx_scaler = StandardScaler()
hetero_tx_scaler.fit(tx_feature_frame.loc[tx_train_mask])
X_transaction = hetero_tx_scaler.transform(tx_feature_frame).astype(np.float32)

addr_scaler = StandardScaler()
addr_scaler.fit(address_feature_frame)
X_address = addr_scaler.transform(address_feature_frame).astype(np.float32)

y = transactions_upgraded['label'].to_numpy(dtype=np.int64)
transaction_ids = transactions_upgraded['txId'].to_numpy()
transaction_to_index = {tx_id: idx for idx, tx_id in enumerate(transaction_ids)}
address_to_index = {address: idx for idx, address in enumerate(address_frame['address'].to_numpy())}


def build_directed_edge_index(frame, src_col, dst_col, src_index, dst_index):
    missing_columns = [col for col in [src_col, dst_col] if col not in frame.columns]
    if missing_columns:
        raise KeyError(f'Missing edge columns: {missing_columns}')
    src = frame[src_col].map(src_index)
    dst = frame[dst_col].map(dst_index)
    valid = src.notna() & dst.notna()
    if not valid.any():
        return torch.empty((2, 0), dtype=torch.long)
    return torch.tensor(
        np.stack([
            src.loc[valid].to_numpy(dtype=np.int64),
            dst.loc[valid].to_numpy(dtype=np.int64),
        ], axis=0),
        dtype=torch.long,
    )


edge_tx_tx = build_directed_edge_index(transaction_edges, 'txId1', 'txId2', transaction_to_index, transaction_to_index)
edge_tx_addr = build_directed_edge_index(tx_addr_edges, 'txId', 'address', transaction_to_index, address_to_index)
edge_addr_tx = build_directed_edge_index(addr_tx_edges, 'address', 'txId', address_to_index, transaction_to_index)
edge_addr_addr = build_directed_edge_index(addr_addr_edges, 'src_address', 'dst_address', address_to_index, address_to_index)

train_mask_list = [bool(value) for value in (tx_train_mask & tx_known_label_mask)]
val_mask_list = [bool(value) for value in (tx_val_mask & tx_known_label_mask)]
test_mask_list = [bool(value) for value in (tx_test_mask & tx_known_label_mask)]

hetero_data = HeteroData()
hetero_data['transaction'].x = torch.tensor(X_transaction, dtype=torch.float32)
hetero_data['transaction'].y = torch.tensor(y, dtype=torch.long)
hetero_data['transaction'].train_mask = torch.tensor(train_mask_list, dtype=torch.bool)
hetero_data['transaction'].val_mask = torch.tensor(val_mask_list, dtype=torch.bool)
hetero_data['transaction'].test_mask = torch.tensor(test_mask_list, dtype=torch.bool)
hetero_data['address'].x = torch.tensor(X_address, dtype=torch.float32)

hetero_data['transaction', 'tx_to_tx', 'transaction'].edge_index = edge_tx_tx
hetero_data['transaction', 'tx_to_addr', 'address'].edge_index = edge_tx_addr
hetero_data['address', 'addr_to_tx', 'transaction'].edge_index = edge_addr_tx
hetero_data['address', 'addr_to_addr', 'address'].edge_index = edge_addr_addr

print(hetero_data)
print(f'Transaction feature dim: {hetero_data["transaction"].x.shape[1]}')
print(f'Address feature dim: {hetero_data["address"].x.shape[1]}')
print(f'Directed tx->tx edges: {edge_tx_tx.shape[1]:,}')
print(f'Directed tx->addr edges: {edge_tx_addr.shape[1]:,}')
print(f'Directed addr->tx edges: {edge_addr_tx.shape[1]:,}')
print(f'Directed addr->addr edges: {edge_addr_addr.shape[1]:,}')


HeteroData(
  transaction={
    x=[203769, 307],
    y=[203769],
    train_mask=[203769],
    val_mask=[203769],
    test_mask=[203769],
  },
  address={ x=[822942, 61] },
  (transaction, tx_to_tx, transaction)={ edge_index=[2, 234355] },
  (transaction, tx_to_addr, address)={ edge_index=[2, 837124] },
  (address, addr_to_tx, transaction)={ edge_index=[2, 477117] },
  (address, addr_to_addr, address)={ edge_index=[2, 2868964] }
)
Transaction feature dim: 307
Address feature dim: 61
Directed tx->tx edges: 234,355
Directed tx->addr edges: 837,124
Directed addr->tx edges: 477,117
Directed addr->addr edges: 2,868,964


## 3. Define the directed HeteroGATv2 model


In [ ]:
class DirectedHeteroGATv2(nn.Module):
    def __init__(self, hidden_channels=128, heads=4, dropout=0.35):
        super().__init__()
        self.dropout = dropout
        self.conv1 = HeteroConv(
            {
                ('transaction', 'tx_to_tx', 'transaction'): GATv2Conv((-1, -1), hidden_channels, heads=heads, concat=False, dropout=dropout, add_self_loops=False),
                ('transaction', 'tx_to_addr', 'address'): GATv2Conv((-1, -1), hidden_channels, heads=heads, concat=False, dropout=dropout, add_self_loops=False),
                ('address', 'addr_to_tx', 'transaction'): GATv2Conv((-1, -1), hidden_channels, heads=heads, concat=False, dropout=dropout, add_self_loops=False),
                ('address', 'addr_to_addr', 'address'): GATv2Conv((-1, -1), hidden_channels, heads=heads, concat=False, dropout=dropout, add_self_loops=False),
            },
            aggr='sum',
        )
        self.conv2 = HeteroConv(
            {
                ('transaction', 'tx_to_tx', 'transaction'): GATv2Conv((-1, -1), hidden_channels, heads=heads, concat=False, dropout=dropout, add_self_loops=False),
                ('transaction', 'tx_to_addr', 'address'): GATv2Conv((-1, -1), hidden_channels, heads=heads, concat=False, dropout=dropout, add_self_loops=False),
                ('address', 'addr_to_tx', 'transaction'): GATv2Conv((-1, -1), hidden_channels, heads=heads, concat=False, dropout=dropout, add_self_loops=False),
                ('address', 'addr_to_addr', 'address'): GATv2Conv((-1, -1), hidden_channels, heads=heads, concat=False, dropout=dropout, add_self_loops=False),
            },
            aggr='sum',
        )
        self.classifier = nn.Linear(hidden_channels, 2)

    def forward(self, x_dict, edge_index_dict):
        x_dict = self.conv1(x_dict, edge_index_dict)
        x_dict = {key: F.elu(value) for key, value in x_dict.items()}
        x_dict = {key: F.dropout(value, p=self.dropout, training=self.training) for key, value in x_dict.items()}
        x_dict = self.conv2(x_dict, edge_index_dict)
        x_dict = {key: F.elu(value) for key, value in x_dict.items()}
        x_dict = {key: F.dropout(value, p=self.dropout, training=self.training) for key, value in x_dict.items()}
        return self.classifier(x_dict['transaction'])


def evaluate_hetero_gatv2(model, batch, mask):
    model.eval()
    with torch.no_grad():
        logits = model(batch.x_dict, batch.edge_index_dict)
        probabilities = np.asarray(
            torch.softmax(logits[mask], dim=1)[:, 1].detach().cpu().tolist(),
            dtype=np.float32,
        )
        predictions = np.asarray(
            logits[mask].argmax(dim=1).detach().cpu().tolist(),
            dtype=np.int64,
        )
        labels = np.asarray(
            batch['transaction'].y[mask].detach().cpu().tolist(),
            dtype=np.int64,
        )

    metrics = {
        'f1': f1_score(labels, predictions, zero_division=0),
        'precision': precision_score(labels, predictions, zero_division=0),
        'recall': recall_score(labels, predictions, zero_division=0),
        'roc_auc': roc_auc_score(labels, probabilities) if len(np.unique(labels)) > 1 else np.nan,
    }
    return metrics, labels, predictions, probabilities


In [ ]:
def train_hetero_gatv2(batch, model_path, hidden_channels=128, heads=4, dropout=0.35, lr=1e-3, weight_decay=5e-4, epochs=60, patience=10):
    model_local = DirectedHeteroGATv2(hidden_channels=hidden_channels, heads=heads, dropout=dropout).to(DEVICE)
    batch = batch.to(DEVICE)
    optimizer_local = torch.optim.Adam(model_local.parameters(), lr=lr, weight_decay=weight_decay)

    train_targets = batch['transaction'].y[batch['transaction'].train_mask]
    class_counts = torch.bincount(train_targets, minlength=2)
    class_weights = (class_counts.sum() / (2.0 * class_counts.clamp_min(1))).to(torch.float32)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))

    history_local = []
    best_epoch_local = -1
    best_val_f1_local = -1.0
    best_state_local = None
    patience_counter_local = 0

    for epoch in range(1, epochs + 1):
        model_local.train()
        optimizer_local.zero_grad()
        logits = model_local(batch.x_dict, batch.edge_index_dict)
        loss = criterion(
            logits[batch['transaction'].train_mask],
            batch['transaction'].y[batch['transaction'].train_mask],
        )
        loss.backward()
        optimizer_local.step()

        train_metrics, _, _, _ = evaluate_hetero_gatv2(model_local, batch, batch['transaction'].train_mask)
        val_metrics, _, _, _ = evaluate_hetero_gatv2(model_local, batch, batch['transaction'].val_mask)
        history_local.append(
            {
                'epoch': epoch,
                'loss': float(loss.item()),
                'train_f1': train_metrics['f1'],
                'val_f1': val_metrics['f1'],
                'val_precision': val_metrics['precision'],
                'val_recall': val_metrics['recall'],
                'val_roc_auc': val_metrics['roc_auc'],
            }
        )

        if val_metrics['f1'] > best_val_f1_local:
            best_val_f1_local = val_metrics['f1']
            best_epoch_local = epoch
            best_state_local = {key: value.detach().cpu().clone() for key, value in model_local.state_dict().items()}
            torch.save(best_state_local, model_path)
            patience_counter_local = 0
        else:
            patience_counter_local += 1

        if epoch == 1 or epoch % 5 == 0:
            print(
                f"Epoch {epoch:03d} | loss={loss.item():.4f} | "
                f"train_f1={train_metrics['f1']:.4f} | val_f1={val_metrics['f1']:.4f} | "
                f"val_auc={val_metrics['roc_auc']:.4f}"
            )

        if patience_counter_local >= patience:
            print(f'Early stopping at epoch {epoch} (best epoch: {best_epoch_local})')
            break

    if best_state_local is not None:
        model_local.load_state_dict(best_state_local)

    val_metrics, val_labels, val_preds, val_probs = evaluate_hetero_gatv2(model_local, batch, batch['transaction'].val_mask)
    test_metrics, test_labels, test_preds, test_probs = evaluate_hetero_gatv2(model_local, batch, batch['transaction'].test_mask)
    return {
        'model': model_local,
        'history': pd.DataFrame(history_local),
        'best_epoch': best_epoch_local,
        'model_path': model_path,
        'val_metrics': val_metrics,
        'val_labels': val_labels,
        'val_preds': val_preds,
        'val_probs': val_probs,
        'test_metrics': test_metrics,
        'test_labels': test_labels,
        'test_preds': test_preds,
        'test_probs': test_probs,
    }


gatv2_train_config = {
    'hidden_channels': 128,
    'heads': 4,
    'dropout': 0.35,
    'lr': 1e-3,
    'weight_decay': 5e-4,
    'epochs': 60,
    'patience': 10,
}

if DEVICE.type == 'cpu':
    gatv2_train_config.update({
        'hidden_channels': 32,
        'heads': 2,
        'epochs': 20,
        'patience': 5,
    })
    print('CPU detected: using a lighter HeteroGATv2 configuration for notebook execution.')

gatv2_result = train_hetero_gatv2(
    hetero_data,
    MODEL_DIR / 'hetero_gatv2_elliptic_plus_plus_best.pt',
    **gatv2_train_config,
)
gatv2_result['history'].tail()


## 4. Evaluate the HeteroGATv2 model


In [ ]:
gatv2_history_df = gatv2_result['history']
gatv2_best_epoch = gatv2_result['best_epoch']
gatv2_model_path = gatv2_result['model_path']
gatv2_val_metrics = gatv2_result['val_metrics']
gatv2_test_metrics = gatv2_result['test_metrics']
gatv2_test_labels = gatv2_result['test_labels']
gatv2_test_preds = gatv2_result['test_preds']

metrics_table = pd.DataFrame(
    [
        {'split': 'validation', **gatv2_val_metrics},
        {'split': 'test', **gatv2_test_metrics},
    ]
)
metrics_table


In [ ]:
print(f'HeteroGATv2 best epoch: {gatv2_best_epoch}')
print(f'HeteroGATv2 model saved to: {gatv2_model_path}')
print('\nHeteroGATv2 test classification report')
print(classification_report(gatv2_test_labels, gatv2_test_preds, target_names=['licit', 'illicit'], zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(gatv2_history_df['epoch'], gatv2_history_df['loss'], color='#4c78a8')
axes[0].axvline(gatv2_best_epoch, color='black', linestyle='--', alpha=0.7)
axes[0].set_title('HeteroGATv2 training loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')

axes[1].plot(gatv2_history_df['epoch'], gatv2_history_df['train_f1'], label='train illicit F1', color='#54a24b')
axes[1].plot(gatv2_history_df['epoch'], gatv2_history_df['val_f1'], label='validation illicit F1', color='#b279a2')
axes[1].axvline(gatv2_best_epoch, color='black', linestyle='--', alpha=0.7)
axes[1].set_title('HeteroGATv2 F1 by epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('F1')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / 'hetero_gatv2_elliptic_pp_training_curves.png', dpi=180, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix(gatv2_test_labels, gatv2_test_preds),
    display_labels=['licit', 'illicit'],
).plot(ax=ax, cmap='Purples', colorbar=False)
ax.set_title('HeteroGATv2 test confusion matrix')
plt.tight_layout()
plt.savefig(FIG_DIR / 'hetero_gatv2_elliptic_pp_confusion_matrix.png', dpi=180, bbox_inches='tight')
plt.show()
